In [ ]:
import sqlite3
import pandas as pd

In [ ]:
conn = sqlite3.connect(r"..\database\processed\earthquakes_API.db") # Connexión a .db
cursor = conn.cursor()

In [ ]:
def run_query(query):
    return pd.read_sql_query(query, conn)

In [ ]:
cursor.execute("""
create table table1 as
select distinct year, geo_country, count(*) total_terremotos
from earthquakes_clean
group by geo_country, year
;
""")

cursor.execute("""
create table table2 as
select distinct year, geo_country, count(*) as total_terremotos_4
from earthquakes_clean
where mag > 4
group by geo_country, year
;
""")

cursor.execute("""
create table table3 as
select distinct year, geo_country, count(*) as micro
from earthquakes_clean
where mag < 2
group by geo_country, year
;
""")

cursor.execute("""
create table table4 as
select distinct year, geo_country, count(*) as menor
from earthquakes_clean
where mag >= 2 and mag < 4
group by geo_country, year
;
""")

cursor.execute("""
create table table5 as
select distinct year, geo_country, count(*) as moderado
from earthquakes_clean
where mag >= 4 and mag < 6
group by geo_country, year
;
""")

cursor.execute("""
create table table6 as
select distinct year, geo_country, count(*) as fuerte
from earthquakes_clean
where mag >= 6
group by geo_country, year
;
""")

cursor.execute("""
create table table7 as
select distinct year, geo_country, max(mag) as max_mag_year
from earthquakes_clean
group by geo_country, year
;
# """)

cursor.execute("""
            create table definitive as
            select a.year, a.geo_country, a.total_terremotos, b.total_terremotos_4, c.micro, d.menor, e.moderado, f.fuerte, g.max_mag_year
            from table1 as a
            left join table2 as b
               on a.year = b.year and a.geo_country = b.geo_country
            left join table3 as c
               on a.year = c.year and a.geo_country = c.geo_country
            left join table4 as d
               on a.year = d.year and a.geo_country = d.geo_country
            left join table5 as e
               on a.year = e.year and a.geo_country = e.geo_country
            left join table6 as f
               on a.year = f.year and a.geo_country = f.geo_country
            left join table7 as g
               on a.year = g.year and a.geo_country = g.geo_country;
""")

query = """
select *
from definitive;
"""

df = run_query(query)
df.to_csv(r"../power bi/tabla_dashboard.csv")
print(df)

cursor.execute("drop table table1")
cursor.execute("drop table table2")
cursor.execute("drop table table3")
cursor.execute("drop table table4")
cursor.execute("drop table table5")
cursor.execute("drop table table6")
cursor.execute("drop table table7")
cursor.execute("drop table definitive")

cursor.execute("Select name from sqlite_master where type='table'")
cursor.fetchall()